# Pharmasense — N-BEATS NCD: Chronic Disease Demand Forecasting
**N-BEATS with trend + seasonality decomposition stacks for 11 NCD/symptomatic conditions.**

- NCD data generated inline using MBQ (Morbidity-Based Quantification) — no upload needed
- Runtime: ~60-90 min on T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abhi-dev99/agtechathon/blob/main/ml/notebooks/04_train_nbeats_ncd.ipynb)

In [ ]:
import subprocess, sys
r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                   capture_output=True, text=True)
print(f'GPU: {r.stdout.strip()}' if r.returncode == 0 else 'No GPU - proceeding anyway (will be slow)')

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'pytorch-forecasting>=1.1.0', 'lightning>=2.2.0',
    'pandas', 'numpy', 'scikit-learn', 'matplotlib', 'seaborn'], check=True)
print('Done.')

In [ ]:
GITHUB_RAW = 'https://raw.githubusercontent.com/abhi-dev99/agtechathon/main/ml/data'
DRIVE_OUTPUT = '/content/drive/MyDrive/pharmasense_models/'
import os
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
import subprocess
subprocess.run(['wget', '-q', '-O', '/content/district_populations.csv',
                f'{GITHUB_RAW}/district_populations.csv'], check=True)
print('Downloaded district_populations.csv')

In [ ]:
# ── Generate NCD/symptomatic weekly data inline (~3-4 min) ───────────────────
import pandas as pd
import numpy as np
import math
from datetime import date, timedelta

np.random.seed(99)
pop_df = pd.read_csv('/content/district_populations.csv')

NSSO_OPD = 1.644
CONDITIONS_CFG = {
    'hypertension':               {'type':'prevalence','rate':0.281,'adult_frac':0.63,'tx_cov':0.55},
    'type2_diabetes':             {'type':'prevalence','rate':0.073,'adult_frac':0.63,'tx_cov':0.60},
    'tuberculosis':               {'type':'prevalence','rate':0.000211,'adult_frac':1.0,'tx_cov':1.0},
    'anaemia':                    {'type':'prevalence','rate':0.571,'adult_frac':0.35,'tx_cov':0.45},
    'viral_fever_urti':           {'type':'opd','share':0.228,'peak_m':8,'amp':0.25,'sec_m':12},
    'acute_gastritis_dyspepsia':  {'type':'opd','share':0.082,'peak_m':5,'amp':0.10},
    'musculoskeletal_pain':       {'type':'opd','share':0.071,'peak_m':1,'amp':0.18},
    'asthma_copd_respiratory':    {'type':'prev_seasonal','rate':0.0313,'adult_frac':1.0,'tx_cov':0.50,'peak_m':12,'amp':0.30},
    'helminth_infestation':       {'type':'ndp','child_frac':0.37,'ndp_cov':0.76,'rounds':2,'ndd':{5,6,7,30,31,32}},
    'uti_bacterial_infections':   {'type':'opd','share':0.050,'peak_m':6,'amp':0.15},
    'maternal_anc_supplements':   {'type':'birth','cbr':0.0197,'gest':40/52,'anc_cov':0.85},
}

def smult(cfg, month, week):
    if cfg['type'] == 'ndp':
        return 5.0 if week in cfg['ndd'] else 0.20
    amp = cfg.get('amp', 0.0); peak = cfg.get('peak_m')
    sec  = cfg.get('sec_m')
    if not peak: return 1.0
    m1 = amp * math.cos(2*math.pi*(month-peak)/12)
    m2 = amp*0.7 * math.cos(2*math.pi*(month-sec)/12) if sec else 0
    return max(0.5, min(1.6, 1.0+m1+m2))

start = date(2018,1,1); end = date(2024,12,31); d = start; weeks=[]
while d <= end:
    iso=d.isocalendar(); weeks.append((d, iso[1])); d+=timedelta(weeks=1)

records = []
for _, row in pop_df.iterrows():
    pop=max(1000,int(row['population'])); state=row['state']; dist=row['district']
    for cname, cfg in CONDITIONS_CFG.items():
        ctype=cfg['type']
        if ctype in ('prevalence','prev_seasonal'):
            base = pop*cfg['rate']*cfg['adult_frac']*cfg['tx_cov']/52.0
        elif ctype == 'opd':
            base = pop*NSSO_OPD*cfg['share']/52.0
        elif ctype == 'ndp':
            base = pop*cfg['child_frac']*cfg['ndp_cov']*cfg['rounds']/52.0
        elif ctype == 'birth':
            base = pop*cfg['cbr']*cfg['gest']*cfg['anc_cov']
        else: continue
        for (wdate, wnum) in weeks:
            sm=smult(cfg, wdate.month, wnum)
            tm=1.02**((wdate.year-2018)+wdate.month/12)
            noise=np.clip(np.random.normal(1.0,0.05),0.85,1.18)
            pts=max(0,int(base*sm*tm*noise))
            records.append({'week_start_date':str(wdate),'state':state,'district':dist,'condition':cname,'patients_est':pts})

ncd_df = pd.DataFrame(records)
ncd_df['week_start_date'] = pd.to_datetime(ncd_df['week_start_date'])
print(f'NCD data: {len(ncd_df):,} rows | {ncd_df.condition.nunique()} conditions | {ncd_df.district.nunique()} districts')
print(ncd_df.groupby('condition')['patients_est'].mean().round(0).to_string())

In [ ]:
# ── Build TimeSeriesDataSet for N-BEATS ───────────────────────────────────────
from pytorch_forecasting import TimeSeriesDataSet, NBeats
from pytorch_forecasting.data import NaNLabelEncoder
from pytorch_forecasting.metrics import SMAPE
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint, LearningRateMonitor

SEQ_LEN  = 26  # 26-week lookback
PRED_LEN = 4   # 4-week forecast
BATCH    = 256
EPOCHS   = 40

df = ncd_df.copy()
df['log_patients'] = np.log1p(df['patients_est'].clip(lower=0))
df = df.sort_values(['district','condition','week_start_date'])
df['group_id']  = df['district'] + '__' + df['condition']
df['time_idx']  = df.groupby('group_id').cumcount()
df['month']     = df['week_start_date'].dt.month.astype(str)

cutoff = df['time_idx'].max() - PRED_LEN

training = TimeSeriesDataSet(
    df[df['time_idx'] <= cutoff],
    time_idx='time_idx', target='log_patients', group_ids=['group_id'],
    min_encoder_length=SEQ_LEN//2, max_encoder_length=SEQ_LEN,
    min_prediction_length=1, max_prediction_length=PRED_LEN,
    time_varying_known_categoricals=['month'],
    time_varying_unknown_reals=['log_patients'],
    add_relative_time_idx=True, add_target_scales=True,
    allow_missing_timesteps=True,
)
validation = TimeSeriesDataSet.from_dataset(training, df, predict=True, stop_randomization=True)
train_dl = training.to_dataloader(train=True,  batch_size=BATCH, num_workers=2)
val_dl   = validation.to_dataloader(train=False, batch_size=BATCH*2, num_workers=2)
print(f'Training samples: {len(training):,} | Val samples: {len(validation):,}')

In [ ]:
# ── Build N-BEATS model ───────────────────────────────────────────────────────
nbeats = NBeats.from_dataset(
    training,
    learning_rate=5e-3,
    weight_decay=1e-2,
    widths=[32, 512],
    backcast_loss_ratio=0.1,
    loss=SMAPE(),
)
print(f'N-BEATS parameters: {nbeats.size()/1e3:.1f}k')

callbacks = [
    EarlyStopping(monitor='val_loss', patience=6, mode='min'),
    ModelCheckpoint(monitor='val_loss', filename='nbeats-best-{epoch}-{val_loss:.4f}', save_top_k=1, mode='min'),
    LearningRateMonitor(),
]
trainer = pl.Trainer(
    max_epochs=EPOCHS, accelerator='gpu', devices=1,
    callbacks=callbacks, gradient_clip_val=0.1, log_every_n_steps=10,
)

In [ ]:
trainer.fit(nbeats, train_dataloaders=train_dl, val_dataloaders=val_dl)
best_nbeats = NBeats.load_from_checkpoint(callbacks[1].best_model_path)
print(f'Best val_loss: {callbacks[1].best_model_score:.4f}')

In [ ]:
# ── Metrics ───────────────────────────────────────────────────────────────────
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

predictions = best_nbeats.predict(val_dl, return_y=True, trainer_kwargs=dict(accelerator='gpu'))
y_pred_log = predictions.output.cpu().numpy().flatten()
y_true_log = predictions.y[0].cpu().numpy().flatten()
y_pred = np.expm1(np.clip(y_pred_log, 0, None))
y_true = np.expm1(np.clip(y_true_log, 0, None))
mask = y_true > 0

mae  = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mape = np.mean(np.abs((y_true[mask]-y_pred[mask])/y_true[mask]))*100 if mask.sum() > 0 else np.nan
r2   = r2_score(y_true, y_pred)
print('=' * 50)
print('  N-BEATS VALIDATION METRICS')
print('=' * 50)
print(f'  SMAPE val loss: {callbacks[1].best_model_score:.4f}')
print(f'  MAE:  {mae:.2f}')
print(f'  RMSE: {rmse:.2f}')
print(f'  MAPE: {mape:.1f}%')
print(f'  R2:   {r2:.4f}')
print('=' * 50)

In [ ]:
# ── Decomposition plots ────────────────────────────────────────────────────────
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

n = min(8, len(validation))
fig, axes = plt.subplots(2, 4, figsize=(22, 9))
for i, ax in enumerate(axes.flat):
    idx = i * (len(validation) // 8)
    try:
        best_nbeats.plot_prediction(predictions.x, predictions.output, idx=min(idx, len(validation)-1),
                                    ax=ax, show_future_observed=True)
        group_name = df.iloc[min(idx, len(df)-1)]['group_id'].split('__')
        ax.set_title(f'{group_name[-1]} | {group_name[0][:12]}', fontsize=8)
    except Exception as e:
        ax.text(0.5, 0.5, str(e)[:60], ha='center', va='center', fontsize=7)

plt.suptitle(f'N-BEATS NCD Forecasts  |  MAPE={mape:.1f}%  |  R2={r2:.3f}', fontweight='bold')
plt.tight_layout()
plt.savefig(DRIVE_OUTPUT + 'nbeats_ncd_forecasts.png', dpi=150)
plt.show()
print('Forecast plots saved.')

In [ ]:
# ── Save ──────────────────────────────────────────────────────────────────────
import torch, shutil; from datetime import datetime
ckpt_path  = callbacks[1].best_model_path
drive_ckpt = DRIVE_OUTPUT + 'nbeats_ncd_best.ckpt'
shutil.copy2(ckpt_path, drive_ckpt)
artifact = {'dataset_parameters': training.get_parameters(),
            'metrics': {'MAE':float(mae),'RMSE':float(rmse),'MAPE_pct':float(mape),'R2':float(r2)},
            'best_val_loss': float(callbacks[1].best_model_score),
            'trained_at': datetime.now().isoformat(),
            'conditions': df['condition'].unique().tolist()}
torch.save(artifact, DRIVE_OUTPUT + 'nbeats_ncd_latest.pt')
from google.colab import files
files.download(DRIVE_OUTPUT + 'nbeats_ncd_latest.pt')
files.download(drive_ckpt)
print(f'MAPE: {mape:.1f}%  |  R2: {r2:.4f}')